<a href="https://colab.research.google.com/github/theocharistr/LLM-comparative-evaluation/blob/main/Comparative_Evaluation_and_Deployment_of_Large_Language_Models.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
"""
Project: Multi-LLM Evaluation & Deployment Readiness Benchmark
Author: <Theocharis Triantafyllidis>

Description:
- Compare multiple instruction-tuned LLMs
- Measure latency, verbosity, and semantic similarity
- Auto-rank models for enterprise deployment decisions
- Provide an interactive Gradio playground

This notebook is designed to simulate real-world model
selection for production environments.
"""


In [ ]:
# ============================================================
# Step 0: Install dependencies
# ============================================================

!pip install -qU \
    transformers \
    accelerate \
    bitsandbytes \
    sentence-transformers \
    peft \
    gradio \
    pandas \
    matplotlib

In [ ]:
# ============================================================
# Step 1: Imports
# ============================================================
import time
import torch
import pandas as pd
import matplotlib.pyplot as plt
import gradio as gr

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig
)

from sentence_transformers import SentenceTransformer, util
torch.set_grad_enabled(False)

In [ ]:
# ============================================================
# Step 2: Define models and prompts
# ============================================================
""" Models under evaluation (Transformers-compatible)
Models are selected to compare diverse architectures (Mistral, Qwen, and Llama-2 based)
across different parameter scales to evaluate trade-offs in reasoning and creativity.
Using the Transformers library with a PyTorch backend for standardized text-generation,
leveraging its optimized pipeline for efficient model loading and GPU acceleration."""
models = [
    "mistralai/Mistral-7B-Instruct-v0.2",
    "HuggingFaceH4/zephyr-7b-beta",
    "NousResearch/Nous-Hermes-2-Mistral-7B-DPO"
]

# Prompt categories testing different behaviors
prompts = {
    "Factual": "Explain the process of mRNA vaccine development.",
    "Creative": "Write a short futuristic story about AI in hospitals.",
    "Medical advice": "Summarize the risks of AI-based diagnosis systems."
}

In [ ]:
# ============================================================
# Step 3: Semantic similarity model
# ============================================================
embedder = SentenceTransformer("all-MiniLM-L6-v2")
""" Why a similarity model?
- Compare meaning, not wording (LLMs phrase the same idea differently)
- Convert text → vectors and use cosine similarity

Model: all-MiniLM-L6-v2
- 384D sentence embeddings (MiniLM, 6 layers)
- Fast (~5K sentences/sec), small (~80MB)
- ~85% STS accuracy (≈ human agreement)
- Best speed/size/accuracy trade-off

Use in this project:
- Check agreement across Mistral, Qwen, Hermes
- High similarity → same facts
- Low similarity → creative divergence

Similarity scale (cosine):
~0.8–1.0 = same meaning
~0.3–0.7 = related
~0.0–0.3 = unrelated"""

In [ ]:
# ============================================================
# Step 4: Model loader (SINGLE SOURCE OF TRUTH)
# ============================================================
def load_model_and_tokenizer(model_name):
    quant_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_use_double_quant=True,
    )

    tokenizer = AutoTokenizer.from_pretrained(
        model_name,
        trust_remote_code=True,
        use_fast=True
    )

    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        quantization_config=quant_config,
        device_map="auto",              # **REQUIRED**
        torch_dtype=torch.float16,
        trust_remote_code=True
    )

    model.eval()
    return tokenizer, model

In [ ]:
## ============================================================
# Step 5: Preload models ONCE
# ============================================================
# IMPORTANT:
# Models must be loaded a single time and reused across prompts.
# Reloading inside the evaluation loop causes extreme slowdowns
# due to repeated disk/CPU offloading on Colab.
# ------------------------------------------------------------
loaded_models = {}
for model_name in models:
    print(f"Loading model once: {model_name}")
    tokenizer, model = load_model_and_tokenizer(model_name)
    loaded_models[model_name] = (tokenizer, model)


In [ ]:
# ============================================================
# Step 6: Inference
# ============================================================

def run_inference(model, tokenizer, prompt, max_new_tokens=100):
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    start = time.time()
    outputs = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        temperature=0.7
    )
    latency = time.time() - start

    text = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return text, latency, len(text.split())

In [ ]:
# ============================================================
# Step 7: Embeddings + ranking
# ============================================================

def compute_embeddings(texts):
    return {k: embedder.encode(v) for k, v in texts.items()}

def rank_models(models, embeddings, latencies, word_counts):
    scores = {}
    for m in models:
        avg_sim = sum(
            util.cos_sim(embeddings[m], embeddings[o]).item()
            for o in models if o != m
        ) / (len(models) - 1)

        scores[m] = avg_sim / (latencies[m] * max(word_counts[m], 1))
    return scores

In [ ]:
# ============================================================
# Step 8: Main evaluation loop
# ============================================================

results_data = []

for prompt_name, prompt_text in prompts.items():
    print(f"\n=== Prompt Category: {prompt_name} ===")

    prompt_results = {}
    latencies = {}
    word_counts = {}

    for model_name in models:
        tokenizer, model = loaded_models[model_name]

        # **ADDED inference_mode for safety + speed**
        with torch.inference_mode():
            text, latency, wc = run_inference(
                model, tokenizer, prompt_text
            )

        prompt_results[model_name] = text
        latencies[model_name] = latency
        word_counts[model_name] = wc

        print(
            f"{model_name} | {wc} words | {latency:.2f}s\n"
            f"{text}\n{'-'*60}"
        )

        results_data.append({
            "Prompt": prompt_name,
            "Model": model_name,
            "Output": text,
            "WordCount": wc,
            "Latency": latency
        })

    embeddings = compute_embeddings(prompt_results)
    scores = rank_models(models, embeddings, latencies, word_counts)

    best_model = max(scores, key=scores.get)
    print(f"✅ Best model for '{prompt_name}': {best_model}")

In [ ]:
# ============================================================
# Step 9: Save + visualize
# ============================================================

df = pd.DataFrame(results_data)
df.to_csv("llm_evaluation_results.csv", index=False)

for prompt_name in prompts:
    subset = df[df["Prompt"] == prompt_name]
    plt.figure(figsize=(8,4))
    plt.bar(subset["Model"], subset["Latency"])
    plt.title(f"Latency per Model — {prompt_name}")
    plt.ylabel("Seconds")
    plt.xticks(rotation=30)
    plt.show()

In [ ]:
# ============================================================
# Step 10: Gradio UI
# ============================================================
def generate(prompt):
    results = []

    scores = {}

    for model_name in models:
        tokenizer, model = loaded_models[model_name]

        with torch.inference_mode():
            inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
            out = model.generate(**inputs, max_new_tokens=150)

        text = tokenizer.decode(out[0], skip_special_tokens=True)

        sim = util.cos_sim(
            embedder.encode(prompt),
            embedder.encode(text)
        ).item()

        scores[model_name] = sim / max(len(text.split()), 1)
        results.append(text)

    best_model = max(scores, key=scores.get)

    # Annotate best model output
    annotated = []
    for model_name, text in zip(models, results):
        prefix = "✅ BEST MODEL\n\n" if model_name == best_model else ""
        annotated.append(prefix + text)

    return annotated


iface = gr.Interface(
    fn=generate,
    inputs=gr.Textbox(label="Prompt"),
    outputs=[
        gr.Textbox(label=models[0]),
        gr.Textbox(label=models[1]),
        gr.Textbox(label=models[2]),
    ],
    title="LLM Multi-Model Playground",
    description="Compare multiple LLMs side by side. Best model is highlighted."
)

iface.launch(share=True)
